# 🔐 TelSec — Telecom Security Audit Framework
**Version:** 1.0.0  |  **Run on:** Google Colab

> ⚠️ **LEGAL NOTICE**: This tool is for authorized security auditing ONLY.  
> You must have **written authorization** from the network operator before running any active tests.  
> Unauthorized use is illegal. By running this notebook, you accept full legal responsibility.


In [ ]:
# ============================================================
# CELL 1: Legal Acceptance & System Package Installation
# ============================================================

ACCEPT_TERMS = True  # Change to False to abort

if not ACCEPT_TERMS:
    raise RuntimeError('You must accept the legal terms before running TelSec.')

print('✅ Legal terms accepted. Installing system packages...')

import subprocess, sys

# Update apt
subprocess.run(['apt-get', 'update', '-q'], check=False)

# Core system packages
packages = [
    'nmap', 'wireshark-common', 'tshark',
    'git', 'curl', 'wget',
    'libssl-dev', 'libffi-dev', 'build-essential'
]
# Telecom packages (graceful failure)
telecom_packages = ['gr-gsm', 'osmocom-nitb']

subprocess.run(['apt-get', 'install', '-y', '-q'] + packages,
               capture_output=True)
for pkg in telecom_packages:
    result = subprocess.run(['apt-get', 'install', '-y', '-q', pkg],
                            capture_output=True)
    status = '✅' if result.returncode == 0 else '⚠️ (not available)'
    print(f'  {pkg}: {status}')

print('\nSystem packages done!')

In [ ]:
# ============================================================
# CELL 2: Python Dependencies & Repo Setup
# ============================================================

!pip install -q streamlit scapy pyshark plotly pandas networkx \
    reportlab jinja2 pyngrok requests PyYAML cryptography \
    paramiko python-dotenv colorama rich click sqlalchemy \
    psutil httpx aiohttp

import os
if not os.path.exists('/content/teleaudit'):
    print('Cloning TelSec repository...')
    # Use local copy if available, else clone from GitHub
    !git clone https://github.com/your-org/teleaudit /content/teleaudit 2>/dev/null || \
     echo 'Repo not on GitHub — using uploaded files'

os.chdir('/content/teleaudit')
print(f'Working directory: {os.getcwd()}')
!mkdir -p reports/ deps/ logs/

print('✅ Environment ready!')

In [ ]:
# ============================================================
# CELL 3: Launch TelSec UI via ngrok Tunnel
# ============================================================
# Set your ngrok auth token (free at https://dashboard.ngrok.com)
NGROK_AUTH_TOKEN = ''  # ← Paste your ngrok token here

import subprocess
import threading
import time
from pyngrok import ngrok, conf

if NGROK_AUTH_TOKEN:
    conf.get_default().auth_token = NGROK_AUTH_TOKEN
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

def run_streamlit():
    subprocess.run([
        'streamlit', 'run', 'app.py',
        '--server.port', '8501',
        '--server.headless', 'true',
        '--browser.gatherUsageStats', 'false',
        '--server.enableCORS', 'false',
        '--server.enableXsrfProtection', 'false',
    ])

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()

# Wait for Streamlit to boot
time.sleep(8)

# Open ngrok tunnel
try:
    tunnel = ngrok.connect(8501)
    print(f'\n🚀 TelSec is LIVE!')
    print(f'🌐 Public URL: {tunnel.public_url}')
    print(f'\nShare this URL to access TelSec from anywhere.')
    print(f'The tunnel stays active as long as this cell is running.')
except Exception as e:
    print(f'ngrok tunnel failed: {e}')
    print('Access locally at: http://localhost:8501')

In [ ]:
# ============================================================
# CELL 4: (Optional) Upload Authorization Config
# ============================================================
# Upload your config/targets.yaml with authorization details

from google.colab import files
import shutil

print('Upload your targets.yaml (optional):')
uploaded = files.upload()
for fname in uploaded:
    shutil.copy(fname, 'config/targets.yaml')
    print(f'Saved {fname} → config/targets.yaml')

In [ ]:
# ============================================================
# CELL 5: Download Generated Reports
# ============================================================
import os, glob
from google.colab import files

reports = glob.glob('reports/*.pdf') + glob.glob('reports/*.html')
if reports:
    for r in reports:
        files.download(r)
    print(f'Downloaded {len(reports)} report(s)')
else:
    print('No reports generated yet. Run tests first via the UI.')